# TimeMeshin: Spatio-Temporal Context Engine for LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Changmaulee/timemeshin/blob/main/examples/TimeMeshin_Colab_Quickstart.ipynb)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

This notebook demonstrates how **TimeMeshin** eliminates temporal hallucinations, out-of-order state corruptions, and future-data leakage in LLM context retrieval.

In [ ]:
# 1. Install TimeMeshin directly from GitHub
!pip install git+https://github.com/Changmaulee/timemeshin.git

## Step 1: Initialize TimeMeshin Engine
Ingest an event sequence where database architecture and team leads evolve over time.

In [ ]:
from timemeshin import TimeMeshinClient
from datetime import datetime, timedelta

client = TimeMeshinClient(db_path='colab_demo.db')
t0 = datetime(2026, 9, 1, 9, 0)

# Day 1: Sarah sets up PostgreSQL with $1,000 budget
client.ingest_event(t0, 'EngineeringTeam', 'TeamLead', 'Sarah', reason='Appointed tech lead')
client.ingest_event(t0, 'Infrastructure', 'Database', 'PostgreSQL', reason='Initial setup')
client.ingest_event(t0, 'Finance', 'MonthlyBudget', '$1,000', reason='Bootstrap budget')

# Day 2: High write loads force migration to DynamoDB ($5,000)
t1 = t0 + timedelta(days=1)
client.ingest_event(t1, 'Infrastructure', 'Database', 'DynamoDB', reason='Write lock contention')
client.ingest_event(t1, 'Finance', 'MonthlyBudget', '$5,000', reason='DynamoDB provisioning')

# Day 3: Leadership transition to Alex
t2 = t0 + timedelta(days=2)
client.ingest_event(t2, 'EngineeringTeam', 'TeamLead', 'Alex', reason='Parental leave')

# Day 4: Cost optimization to Postgres + Redis ($1,800)
t3 = t0 + timedelta(days=3)
client.ingest_event(t3, 'Infrastructure', 'Database', 'Postgres + Redis Cache', reason='Cost optimization')
client.ingest_event(t3, 'Finance', 'MonthlyBudget', '$1,800', reason='Optimized cache tier')

print('Ingestion complete!')

## Step 2: Time-Travel Playhead Scrubbing
Query the ground-truth state at **Day 2 (12:00)** before Day 4 occurred.

In [ ]:
query_time = datetime(2026, 9, 2, 12, 0)
result = client.query_at(query_time, query='database budget lead')

print('=' * 60)
print(f'TIMEMESHIN GROUND TRUTH AT {query_time}:')
print('=' * 60)
for k, v in result['state'].items():
    print(f'* {k}: {v}')
print('=' * 60)
print('Transitive Causal Trail:')
print(result['causal_summary'])
